# Exploration des failles de sécurité du NVD

## Cadrage

Je fais l'audit cybersécurité de la société de services informatiques lyonnaise "Yggdrasil Systems". Chaque département est audité et je suis chargée de la partie cybersécurité.

**Les questions du client**
- Les failles de sécurité publiées deviennent-elles plus nombreuses, plus graves ou plus faciles à exploiter ?
- Sur quels éditeurs et quels logiciels concentrer la vigilance ?

**Comment je vais y répondre**
- Le nombre de failles publiées dans le temps.
- La gravité des failles.
- Les critères d'attaque (origine, difficulté, compte nécessaire, action de la victime).
- Les éditeurs et produits touchés et les types de défaut.

**Les données**
- Source. L'API du NVD, la base américaine des failles de sécurité.
- Période. Du 1er octobre 2025 au 16 octobre 2026.
- Fichier. `nvd_failles.csv`, avec 96 765 failles et 27 colonnes.

**Limite à garder en tête**
On mesure les failles publiées. On ne mesure pas les attaques réellement subies.

Ce notebook sert à explorer les données pour décider comment les nettoyer et quelle version de la note de gravité utiliser (v3.1 ou v4.0).

## 1. Import et premières vérifications

In [1]:
# Je charge pandas et je lis le CSV en convertissant les deux colonnes de dates.
import pandas as pd

df = pd.read_csv("../data/nvd_failles.csv", parse_dates=["published", "last_modified"])
print("Forme du tableau (lignes, colonnes) :", df.shape)

Forme du tableau (lignes, colonnes) : (96765, 27)


### Types des colonnes et aperçu

Je vérifie que chaque colonne a le bon type et je regarde les premières lignes.

In [2]:
df.info()
df.head(3).T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96765 entries, 0 to 96764
Data columns (total 27 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   cve_id                      96765 non-null  object        
 1   published                   96765 non-null  datetime64[ns]
 2   last_modified               96765 non-null  datetime64[ns]
 3   vuln_status                 96765 non-null  object        
 4   base_score_v31              84602 non-null  float64       
 5   base_severity_v31           84602 non-null  object        
 6   attack_vector_v31           84602 non-null  object        
 7   attack_complexity_v31       84602 non-null  object        
 8   privileges_required_v31     84602 non-null  object        
 9   user_interaction_v31        84602 non-null  object        
 10  confidentiality_impact_v31  84602 non-null  object        
 11  integrity_impact_v31        84602 non-null  object    

,0,1,2
cve_id,CVE-2025-61714,CVE-2025-61715,CVE-2025-61716
published,2025-10-01 03:15:36.940000,2025-10-01 03:15:37.037000,2025-10-01 03:15:37.113000
last_modified,2026-10-15 20:10:00.197000,2026-10-15 20:10:00.197000,2026-10-15 20:10:00.197000
vuln_status,Rejected,Rejected,Rejected
base_score_v31,NaN,NaN,NaN
base_severity_v31,NaN,NaN,NaN
attack_vector_v31,NaN,NaN,NaN
attack_complexity_v31,NaN,NaN,NaN
privileges_required_v31,NaN,NaN,NaN
user_interaction_v31,NaN,NaN,NaN


## 2. EDA

L'EDA sert à décider comment nettoyer les données dans le script final. Chaque partie correspond à une décision.

| # | Décision pour le script final | Ce que je regarde |
|---|---|---|
| 1 | Version de la note faille à garder (v3.1 ou v4.0) | Part de failles avec chaque version dans le temps |
| 2 | Failles sans note dans la version choisie | Combien et lesquelles |
| 3 | Failles rejetées | Combien et ce qu'elles contiennent |
| 4 | Notes Primary ou Secondary | Part de chaque type dans le temps |
| 5 | Éditeurs et produits | Valeurs manquantes, doublons, variantes d'écriture |
| 6 | Types de défaut | Valeurs vagues, failles sans type de défaut |
| 7 | Scores à 0.0 | Combien et lesquels |


### 1. Version de la note à garder

In [3]:
# Je calcule pour chaque mois de publication la part de failles avec une note v3.1 et avec une note v4.0.
mois = df["published"].dt.to_period("M")
couverture = pd.DataFrame({
    "nb_failles": df.groupby(mois).size(),
    "part_v31_%": (df["base_score_v31"].notna().groupby(mois).mean() * 100).round(1),
    "part_v40_%": (df["base_score_v40"].notna().groupby(mois).mean() * 100).round(1),
})
couverture

,nb_failles,part_v31_%,part_v40_%
published,,,
2025-10,4389,88.4,31.1
2025-11,3116,88.8,27.3
2025-12,5643,80.2,24.9
2026-01,5143,77.9,28.6
2026-02,4808,91.1,41.3
2026-03,6304,96.3,36.8
2026-04,5885,95.7,36.4
2026-05,6985,93.4,32.6
2026-06,8001,92.0,29.6


La note v3.1 est présente sur 78 % à 96 % des failles selon le mois. La note v4.0 est présente sur 25 % à 41 % des failles selon le mois. La v3.1 couvre plus de failles que la v4.0 tous les mois.

La part de v4.0 n'augmente pas avec le temps. Elle monte en février 2026 (41 %) puis redescend jusqu'à 25 % en juillet 2026 et remonte à 33 % en octobre 2026. L'idée que la v4.0 serait plus présente sur les failles récentes n'est pas confirmée par ces données.

Sur toute la période 12 163 failles n'ont pas de note v3.1 (13 %). 67 264 failles n'ont pas de note v4.0 (70 %).

In [4]:
# Je compte les failles selon qu'elles ont une note v3.1, une note v4.0, les deux ou aucune.
a_v31 = df["base_score_v31"].notna()
a_v40 = df["base_score_v40"].notna()
pd.crosstab(a_v31, a_v40, rownames=["note v3.1"], colnames=["note v4.0"], margins=True)

note v4.0,False,True,All
note v3.1,,,
False,6371,5792,12163
True,60893,23709,84602
All,67264,29501,96765


Sur les 96 765 failles.
- 23 709 ont les deux notes (24,5 %).
- 60 893 ont seulement la note v3.1 (62,9 %).
- 5 792 ont seulement la note v4.0 (6,0 %).
- 6 371 n'ont aucune note (6,6 %).

Si je garde la v3.1 je n'ai pas de note pour 12 163 failles (12,6 %). Si je garde la v4.0 je n'ai pas de note pour 67 264 failles (69,5 %). Les failles sans aucune note le restent quelle que soit la version.

**Conclusion** 
Je garde la version v3.1 de la note pour tout le dashboard. Je n'utilise pas la v4.0.

Pourquoi.
- La v3.1 est présente sur 87 % des failles. La v4.0 l'est sur 30 %.
- La part de failles avec une note v3.1 reste entre 78 % et 96 % chaque mois. Pour la v4.0 elle varie entre 25 % et 41 %. Je peux donc comparer les mois entre eux.

Ce que je perds.
- 5 792 failles ont seulement une note v4.0 (6 %).
- Le détail de l'action de la victime en v4.0 (trois valeurs au lieu de deux) et les conditions d'attaque. Mes graphiques prévus restent possibles avec la v3.1.

### 2. Failles sans note v3.1

12 163 failles n'ont pas de note v3.1 (13 %). Je regarde qui elles sont, en commençant par leur statut.

In [7]:
# Je regarde pour chaque statut combien de failles n'ont pas de note v3.1.
sans_note = df["base_score_v31"].isna()
resume = df.groupby("vuln_status").agg(nb_failles=("cve_id", "size"))
resume["sans_v31"] = sans_note.groupby(df["vuln_status"]).sum()
resume["part_sans_v31_%"] = (resume["sans_v31"] / resume["nb_failles"] * 100).round(1)
resume

,nb_failles,sans_v31,part_sans_v31_%
vuln_status,,,
Analyzed,38613,258,0.7
Awaiting Analysis,7380,1482,20.1
Deferred,36093,5340,14.8
Modified,5918,4,0.1
Received,4999,2380,47.6
Rejected,2458,2458,100.0
Undergoing Analysis,1304,241,18.5


Toutes les failles rejetées sont sans note. Le manque de note dépend du statut. Seulement 0,7 % des failles Analyzed n'ont pas de note v3.1. C'est 47,6 % des failles Received.

Le nombre de failles rejetées (2 458) est le même que le nombre de failles sans produit. Il reste à vérifier si ce sont les mêmes failles.

In [8]:
# Je regarde par mois la part de failles sans note v3.1 en laissant de côté les failles rejetées.
non_rejetees = df[df["vuln_status"] != "Rejected"]
mois_nr = non_rejetees["published"].dt.to_period("M")
(non_rejetees["base_score_v31"].isna().groupby(mois_nr).mean() * 100).round(1)

published
2025-10     9.2
2025-11     8.4
2025-12    17.6
2026-01     6.9
2026-02     5.1
2026-03     2.6
2026-04     3.1
2026-05     5.9
2026-06     7.3
2026-07     8.0
2026-08    14.6
2026-09    17.2
2026-10    12.5
Freq: M, Name: base_score_v31, dtype: float64

Hors failles rejetées 9 705 failles sur 94 307 n'ont pas de note v3.1 (10,3 %).

La part varie beaucoup selon le mois. Elle va de 2,6 % en mars 2026 à 17,6 % en décembre 2025. Elle monte de 3,1 % en avril 2026 à 17,2 % en septembre 2026. Les mois récents ont donc plus de failles sans note.

Ce n'est pas seulement une question de récence. Octobre et novembre 2025 sont à 9,2 % et 8,4 %.

**Conclusion**
Je garde les failles sans note v3.1 dans le tableau.
- Je les utilise pour les comptages (volume par mois, éditeurs, types de défaut).
- Je ne les utilise pas pour les graphiques de gravité et de critères d'attaque. Sans note elles n'ont pas de gravité.

Les failles rejetées seront exclues partout (voir décision 3).

### 3. Failles rejetées

Les failles rejetées sont 2 458. Je regarde ce qu'elles contiennent et si ce sont les mêmes que les failles sans produit.

In [10]:
# Je regarde ce que contiennent les failles rejetées et si ce sont les mêmes que celles sans produit.
rejetees = df["vuln_status"] == "Rejected"
sans_produit = df["produits"].isna()
sans_cwe = df["cwes"].isna()
print("Rejetées :", rejetees.sum())
print("Rejetées sans produit :", (rejetees & sans_produit).sum())
print("Sans produit mais non rejetées :", (~rejetees & sans_produit).sum())
print("Rejetées sans type de défaut :", (rejetees & sans_cwe).sum())

Rejetées : 2458
Rejetées sans produit : 2458
Sans produit mais non rejetées : 0
Rejetées sans type de défaut : 2458


Ce sont donc les mêmes failles que celles sans produit. La doc du NVD dit que les données associées sont retirées quand une faille est rejetée.

**Conclusion**
J'exclus les failles rejetées de toutes les analyses (comptages comme graphiques).
- Elles n'ont ni note ni produit ni type de défaut.
- Elles sont 2 458 sur 96 765 (2,5 %). Il reste 94 307 failles.

### 4. Notes Primary ou Secondary

Une faille peut avoir plusieurs notes v3.1. La note Primary vient du NVD lui-même. Une note Secondary vient d'une autre organisation (par exemple l'organisme qui a déclaré la faille). Pour chaque faille je garde la note Primary si elle existe. Sinon je garde la première note Secondary.

Je regarde ici seulement les notes v3.1. Je veux savoir quelle part de chaque type j'ai dans le temps et si les deux types donnent des gravités différentes.

In [13]:
# Je compte les notes v3.1 Primary et Secondary et je calcule la part de Primary par mois.
d31 = df[df["type_note_v31"].notna()]
print(d31["type_note_v31"].value_counts())
mois31 = d31["published"].dt.to_period("M")
(d31["type_note_v31"].eq("Primary").groupby(mois31).mean() * 100).round(1)

type_note_v31
Secondary    66345
Primary      18257
Name: count, dtype: int64


published
2025-10    41.8
2025-11    27.9
2025-12    28.2
2026-01    32.9
2026-02    42.3
2026-03    39.8
2026-04    29.2
2026-05    27.8
2026-06    18.7
2026-07    15.7
2026-08     8.6
2026-09     8.1
2026-10    11.1
Freq: M, Name: type_note_v31, dtype: float64

Sur les 84 602 failles avec une note v3.1, 18 257 ont une note Primary (22 %). 66 345 ont une note Secondary (78 %).

La part de notes Primary baisse dans le temps. Elle est d'environ 42 % en octobre 2025 et en février 2026. Elle tombe à environ 8 % en août et en septembre 2026. Elle remonte à 11 % en octobre 2026.

In [14]:
# Je compare les gravités v3.1 des notes Primary et Secondary (répartition en % et score moyen).
print((pd.crosstab(d31["type_note_v31"], d31["base_severity_v31"], normalize="index") * 100).round(1))
d31.groupby("type_note_v31")["base_score_v31"].agg(["count", "mean", "median"]).round(2)

base_severity_v31  CRITICAL  HIGH  LOW  MEDIUM  NONE
type_note_v31                                       
Primary                16.0  36.1  1.8    46.2   0.0
Secondary              11.7  44.7  3.5    40.0   0.0


,count,mean,median
type_note_v31,,,
Primary,18257,7.03,7.1
Secondary,66345,7.01,7.2


Les deux types ont presque le même score. Le score moyen est de 7,03 pour les notes Primary et de 7,01 pour les notes Secondary. Les médianes sont de 7,1 et 7,2.

Les notes Secondary sont plus souvent High. Les notes Primary sont plus souvent Critical ou Medium.

Chaque faille a une seule note dans mon tableau. Je compare donc des failles différentes. L'écart peut venir des failles et pas seulement du type de note.

**Conclusion**

Je garde la règle. Pour chaque faille je prends la note v3.1 Primary si elle existe. Sinon je prends la première note Secondary.

Pourquoi.
- La note Primary vient d'un seul évaluateur (le NVD) avec la même méthode pour toutes les failles.
- Les notes Secondary viennent de plusieurs organisations qui peuvent noter autrement.

### 5. Éditeurs et produits

La colonne `produits` contient plusieurs produits par faille, séparés par des `;`. Je l'éclate pour avoir une ligne par faille et par produit. Je laisse de côté les failles rejetées.

In [15]:
# J'éclate la colonne produits (hors failles rejetées) pour avoir une ligne par faille et par produit.
base = df.loc[df["vuln_status"] != "Rejected", ["cve_id", "produits"]]
produits = base.assign(produit=base["produits"].str.split(";")).explode("produit")
print("Lignes :", len(produits))
print("Failles :", produits["cve_id"].nunique())

# Je vérifie que chaque morceau contient un seul séparateur éditeur|produit.
print(produits["produit"].str.count(r"\|").value_counts())

Lignes : 252643
Failles : 94307
produit
1    252469
2        96
0        68
3         5
4         3
5         2
Name: count, dtype: int64


Une fois éclatée, la colonne donne 252 643 lignes pour 94 307 failles. Il y a en moyenne environ 2,7 produits par faille.

Dans 252 469 cas (99,9 %) le morceau contient bien un seul séparateur entre l'éditeur et le produit. Pour 174 morceaux ce n'est pas le cas. Il y en a 68 sans séparateur et 106 avec plusieurs séparateurs. Je dois regarder des exemples pour comprendre.

In [16]:
# Je regarde des exemples de morceaux qui n'ont pas exactement un séparateur.
nb_barres = produits["produit"].str.count(r"\|")
for morceau in produits.loc[nb_barres != 1, "produit"].head(15):
    print(morceau)

cyberlord92|Password Policy Manager | Password Manager
Rainbow-Themes|Education WordPress Theme | HiStudy
lovelightplugins|Ninja Countdown | Fastest Countdown Builder
sanderkah|Convert WebP & AVIF | Quicq | Best image optimizer and compression plugin | Improve your Google Pagespeed
sundayfanz|wModes – Catalog Mode, Product Pricing, Enquiry Forms & Promotions | for WooCommerce
admintwentytwenty|UiPress lite | Effortless custom dashboards, admin themes and pages
admintwentytwenty|UiPress lite | Effortless custom dashboards, admin themes and pages
surbma|Surbma | MiniCRM Shortcode
admintwentytwenty|UiPress lite | Effortless custom dashboards, admin themes and pages
nalam-1|Magical Shop Builder – WooCommerce Template Builder for Elementor | Shop, Cart, Checkout & Product Page Builder
UBICOD Co., Ltd. | MEDIVISION INC.|UBICOD Medivision Digital Signage
UBICOD Co., Ltd. | MEDIVISION INC.|UBICOD Medivision Digital Signage
emplibot|Emplibot – AI Content Writer with Keyword Research, Infographi

Sur les 15 exemples affichés le séparateur `|` se trouve dans le nom lui-même. C'est surtout le nom du produit qui contient `|` (des titres longs de plugins ou de thèmes). Dans un cas (UBICOD) c'est le nom de l'éditeur.

Je couperai à la première barre seulement. Le nom du produit reste ainsi entier dans la plupart des cas. L'éditeur est mal coupé quand son propre nom contient une barre. Moins de 0,1 % des lignes ont un séparateur en trop ou en moins.

In [17]:
# Je regarde les morceaux qui n'ont aucun séparateur.
for morceau in produits.loc[nb_barres == 0, "produit"].head(15):
    print(morceau)


AMD Ryzen™ 7035 Series Processors with Radeon™ Graphics

AMD Ryzen™ 8000 Series Desktop Processors

AMD Ryzen™ 9000 Series Desktop Processors

AMD Ryzen™ 7035 Series Processors with Radeon™ Graphics

AMD Ryzen™ 8000 Series Desktop Processors

AMD Ryzen™ 9000 Series Desktop Processors

AMD Ryzen™ 8040 Series Mobile Processors with Radeon™ Graphics
 Display &amp
 Edit Ninja Forms Submissions on your site frontend
 Compress / Optimize Images &amp
s WordCents adSense Widget with Analytics
 Save Entries, File Upload &amp
 Country Code Field
 Motive Commerce Search
 AI Chatbot, Customer Support, FAQ &amp


Les morceaux sans séparateur que j'ai regardés (15 sur 68) sont des bouts de noms coupés. Un `;` se trouve dans le nom lui-même et a coupé le nom en deux.

Deux cas dans les exemples.
- Des noms de plugins avec `&amp;`. C'est le code HTML du signe `&`. Son `;` a coupé le nom.
- Des listes de processeurs AMD. Elles ressemblent à un seul nom de produit qui contient plusieurs lignes séparées par des `;`.

Ces morceaux ne contiennent pas d'éditeur. Je les retire. Le début du nom qui les précède reste dans le tableau mais coupé. Ça concerne très peu de lignes.

Les noms contiennent aussi du code HTML. À décoder dans le script final.

In [18]:
# Je sépare l'éditeur et le produit (première barre) et je regarde les 10 éditeurs avec le plus de failles.
valides = produits[nb_barres >= 1].reset_index(drop=True)
valides[["vendor", "product"]] = valides["produit"].str.split("|", n=1, expand=True)
valides["vendor"] = valides["vendor"].str.strip()
valides["product"] = valides["product"].str.strip()
print("Lignes gardées :", len(valides))
print("Lignes retirées :", len(produits) - len(valides))
print("vendor vides :", (valides["vendor"] == "").sum(), "| product vides :", (valides["product"] == "").sum())
valides.groupby("vendor")["cve_id"].nunique().sort_values(ascending=False).head(10)

Lignes gardées : 252575
Lignes retirées : 68
vendor vides : 249 | product vides : 269


vendor
Linux                         9232
n/a                           5213
Microsoft                     4727
Google                        3871
Oracle Corporation            3155
Red Hat                       2304
Unknown                       1813
IBM                           1543
Adobe                         1207
Apache Software Foundation    1111
Name: cve_id, dtype: int64

Après la séparation il reste 252 575 lignes. 68 ont été retirées. Il y a 249 éditeurs vides et 269 produits vides.

`n/a` et `Unknown` sont dans le top 10. Ce sont des éditeurs manquants. Avec les 249 éditeurs vides, ça fait au moins trois façons d'écrire un éditeur manquant.

In [19]:
# Je regarde les éditeurs écrits de plusieurs façons avec des majuscules et minuscules différentes.
valides["vendor_min"] = valides["vendor"].str.lower()
print("Éditeurs distincts :", valides["vendor"].nunique())
print("Éditeurs distincts en minuscules :", valides["vendor_min"].nunique())
variantes = valides.groupby("vendor_min")["vendor"].nunique()
variantes[variantes > 1].sort_values(ascending=False).head(10)

Éditeurs distincts : 12758
Éditeurs distincts en minuscules : 12148


vendor_min
nsauditor       4
wpvibes         3
github          3
php-fusion      3
wbce            3
dataease        3
wpmet           3
deltaww         3
opencats        3
designthemes    3
Name: vendor, dtype: int64

Il y a 12 758 éditeurs différents. En minuscules il en reste 12 148. 610 écritures sont donc des variantes de casse.

Dans la liste affichée un même éditeur est écrit de 3 à 4 façons.

In [20]:
# Je regarde le top 10 en minuscules puis les écritures de n/a et unknown puis celles de microsoft.
print(valides.groupby("vendor_min")["cve_id"].nunique().sort_values(ascending=False).head(10))
print(valides.loc[valides["vendor_min"].isin(["n/a", "unknown"]), "vendor"].value_counts())
print(valides.loc[valides["vendor_min"].str.contains("microsoft"), "vendor"].value_counts().head(10))

vendor_min
linux                         9232
n/a                           5216
microsoft                     4781
google                        3871
oracle corporation            3155
red hat                       2304
unknown                       1813
ibm                           1543
adobe                         1207
apache software foundation    1111
Name: cve_id, dtype: int64
vendor
n/a        5232
Unknown    1848
N/A           3
Name: count, dtype: int64
vendor
Microsoft         56116
microsoft            63
microsoftstart        1
Name: count, dtype: int64


Le top 10 en minuscules est le même que le top 10 d'avant. L'ordre ne change pas. Seuls deux chiffres bougent. microsoft passe de 4 727 à 4 781 failles et n/a de 5 213 à 5 216.

Un éditeur manquant s'écrit n/a (5 232 lignes) ou Unknown (1 848 lignes) ou N/A (3 lignes). Il peut aussi être vide (249 lignes).

Pour Microsoft il y a trois écritures. Microsoft (56 116 lignes) et microsoft (63 lignes) ne diffèrent que par la casse. La troisième est microsoftstart (1 ligne). Ça fait environ 12 lignes par faille pour Microsoft. Les lignes ne sont donc pas un bon moyen de compter. Je compte des failles.

In [21]:
# Je compte les lignes en double (même faille, même éditeur, même produit).
doublons = valides.duplicated(subset=["cve_id", "vendor_min", "product"])
print("Lignes en double :", doublons.sum(), "sur", len(valides))
print("Failles concernées :", valides.loc[doublons, "cve_id"].nunique())

Lignes en double : 57499 sur 252575
Failles concernées : 11885


57 499 lignes sont en double sur 252 575. Une ligne est en double quand la même faille cite le même éditeur et le même produit. 11 885 failles sont concernées.

Sans les doublons il reste 195 076 lignes.

Je ne regroupe que les variantes de casse. Un éditeur écrit avec un autre nom (par exemple Oracle et Oracle Corporation) reste séparé. Je n'ai pas vérifié si ça arrive.

**conclusion**

Règles de nettoyage pour les éditeurs et les produits dans le script final.
- Écrire une vraie table avec une ligne par produit. Les listes jointes par des `;` coupent des noms (174 cas).
- Décoder le code HTML des noms (`&amp;` devient `&`).
- Retirer les espaces au début et à la fin des noms.
- Un éditeur vide ou écrit n/a ou N/A ou Unknown devient « non renseigné ». Il n'entre pas dans les classements.
- Un produit vide devient « non renseigné ». La ligne est gardée.
- Regrouper les éditeurs sur leur écriture en minuscules et garder l'écriture d'origine à côté.
- Supprimer les lignes en double (même faille, même éditeur, même produit). Il reste alors 195 076 lignes sur 252 575.
- Compter des failles distinctes par éditeur. Jamais des lignes.

### 6. Types de défaut

La colonne `cwes` contient les types de défaut (codes CWE) de chaque faille, séparés par des `;`. Je l'éclate pour avoir une ligne par faille et par type de défaut. Je laisse de côté les failles rejetées.

In [22]:
# J'éclate la colonne cwes (hors failles rejetées) pour avoir une ligne par faille et par type de défaut.
base_cwe = df.loc[df["vuln_status"] != "Rejected", ["cve_id", "cwes"]].dropna(subset=["cwes"])
cwe = base_cwe.assign(cwe=base_cwe["cwes"].str.split(";")).explode("cwe").reset_index(drop=True)
print("Lignes :", len(cwe))
print("Failles avec au moins un type de défaut :", cwe["cve_id"].nunique())
print("Failles sans type de défaut (hors rejetées) :", (df["vuln_status"] != "Rejected").sum() - cwe["cve_id"].nunique())
cwe.groupby("cwe")["cve_id"].nunique().sort_values(ascending=False).head(10)

Lignes : 106783
Failles avec au moins un type de défaut : 88139
Failles sans type de défaut (hors rejetées) : 6168


cwe
CWE-79            9226
CWE-862           5174
CWE-89            4672
CWE-284           4227
CWE-416           3090
CWE-22            3046
CWE-125           2418
NVD-CWE-noinfo    2365
CWE-74            2324
CWE-78            2264
Name: cve_id, dtype: int64

Hors failles rejetées 88 139 failles ont au moins un type de défaut. 6 168 n'en ont aucun (6,5 %). Une faille qui a un type de défaut en a en moyenne environ 1,2.

NVD-CWE-noinfo est dans le top 10.

In [23]:
# Je compte les types de défaut vagues et je vérifie le format des codes.
vague = cwe["cwe"].isin(["NVD-CWE-Other", "NVD-CWE-noinfo"])
print("Failles avec NVD-CWE-noinfo :", cwe.loc[cwe["cwe"] == "NVD-CWE-noinfo", "cve_id"].nunique())
print("Failles avec NVD-CWE-Other :", cwe.loc[cwe["cwe"] == "NVD-CWE-Other", "cve_id"].nunique())
seulement_vague = (~vague).groupby(cwe["cve_id"]).sum() == 0
print("Failles qui n'ont que des types vagues :", seulement_vague.sum())
print("Codes vides :", (cwe["cwe"] == "").sum())
print("Codes au format inattendu :", (~cwe["cwe"].str.match(r"^(CWE-\d+|NVD-CWE-Other|NVD-CWE-noinfo)$")).sum())

Failles avec NVD-CWE-noinfo : 2365
Failles avec NVD-CWE-Other : 180
Failles qui n'ont que des types vagues : 1437
Codes vides : 0
Codes au format inattendu : 0


Tous les codes ont un format normal. Il n'y a aucun code vide.

Deux types sont vagues. NVD-CWE-noinfo est dans 2 365 failles. NVD-CWE-Other est dans 180 failles.

1 437 failles n'ont que des types vagues. Avec les 6 168 failles sans type de défaut ça fait 7 605 failles sans type de défaut précis (8,1 %).

In [24]:
# Je compte les failles avec un type vague et un vrai type et les lignes en double.
a_vague = vague.groupby(cwe["cve_id"]).any()
a_vrai = (~vague).groupby(cwe["cve_id"]).any()
print("Failles avec un type vague et un vrai type :", (a_vague & a_vrai).sum())
doublons_cwe = cwe.duplicated(subset=["cve_id", "cwe"])
print("Lignes en double :", doublons_cwe.sum(), "sur", len(cwe))

Failles avec un type vague et un vrai type : 1108
Lignes en double : 2509 sur 106783


1 108 failles ont un type vague et un vrai type de défaut. 2 509 lignes sont en double (2,3 %). Une ligne est en double quand la même faille cite deux fois le même type de défaut.

NVD-CWE-noinfo et NVD-CWE-Other ne sont jamais dans la même faille. 2 365 plus 180 donne 2 545 failles. C'est aussi 1 437 failles avec seulement des types vagues plus 1 108 failles avec un type vague et un vrai type.

**Conclusion**

Règles de nettoyage pour les types de défaut dans le script final.
- Écrire une vraie table avec une ligne par faille et par type de défaut.
- Supprimer les lignes en double (même faille et même type). Il reste alors 104 274 lignes sur 106 783.
- NVD-CWE-noinfo et NVD-CWE-Other deviennent « non précisé ».
- Si une faille a un vrai type et un type vague je retire la ligne vague. 1 108 failles sont concernées.
- Une faille sans aucun type de défaut est aussi « non précisé ». 6 168 failles sont concernées.
- Compter des failles distinctes par type de défaut. Jamais des lignes.
- Ajouter une table `cwe_noms` (code et nom) depuis le catalogue MITRE. Les noms ne sont pas dans l'API.

Résultat. « Non précisé » veut dire que la faille n'a aucun type de défaut précis. Cela représente 7 605 failles (8,1 %).

### 7. Scores à 0.0

Certaines failles ont un score v3.1 à 0. Je regarde combien il y en a et lesquelles. Je laisse de côté les failles rejetées.

In [25]:
# Je regarde la distribution des scores v3.1 puis les failles avec un score à 0 (hors failles rejetées).
print(df["base_score_v31"].describe())
nuls = df[(df["base_score_v31"] == 0) & (df["vuln_status"] != "Rejected")]
print("Failles avec un score v3.1 à 0 :", len(nuls))
print(nuls["base_severity_v31"].value_counts(dropna=False))
print(nuls["type_note_v31"].value_counts())
print(nuls["vuln_status"].value_counts())
nuls[["cve_id", "published", "vuln_status", "type_note_v31", "attack_vector_v31"]].head(10)

count    84602.000000
mean         7.011437
std          1.696431
min          0.000000
25%          5.500000
50%          7.200000
75%          8.100000
max         10.000000
Name: base_score_v31, dtype: float64
Failles avec un score v3.1 à 0 : 8
base_severity_v31
NONE    8
Name: count, dtype: int64
type_note_v31
Secondary    8
Name: count, dtype: int64
vuln_status
Deferred             6
Awaiting Analysis    2
Name: count, dtype: int64


,cve_id,published,vuln_status,type_note_v31,attack_vector_v31
36347,CVE-2026-20188,2026-05-06 17:16:21.190,Awaiting Analysis,Secondary,NETWORK
37548,CVE-2026-42873,2026-05-11 20:25:43.833,Deferred,Secondary,NETWORK
48358,CVE-2026-55611,2026-06-24 18:17:19.443,Deferred,Secondary,NETWORK
52246,CVE-2026-57501,2026-07-09 23:17:05.353,Awaiting Analysis,Secondary,NETWORK
70097,CVE-2026-54920,2026-08-25 01:16:36.290,Deferred,Secondary,NETWORK
72353,CVE-2026-55891,2026-08-28 22:16:51.957,Deferred,Secondary,NETWORK
79727,CVE-2026-55630,2026-09-15 16:17:14.850,Deferred,Secondary,NETWORK
82643,CVE-2026-49292,2026-09-17 19:16:48.960,Deferred,Secondary,NETWORK


Les scores v3.1 vont de 0 à 10. Le score moyen est de 7,0 et la médiane de 7,2. Aucune valeur n'est en dehors de l'échelle.

8 failles ont un score à 0 (hors failles rejetées). Elles ont toutes la gravité NONE et une note Secondary. Six sont au statut Deferred et deux sont au statut Awaiting Analysis. Elles ont été publiées entre mai et septembre 2026.

In [26]:
# Je regarde les impacts des 8 failles avec un score à 0.
nuls[["cve_id", "confidentiality_impact_v31", "integrity_impact_v31", "availability_impact_v31"]]

,cve_id,confidentiality_impact_v31,integrity_impact_v31,availability_impact_v31
36347,CVE-2026-20188,NONE,NONE,NONE
37548,CVE-2026-42873,NONE,NONE,NONE
48358,CVE-2026-55611,NONE,NONE,NONE
52246,CVE-2026-57501,NONE,NONE,NONE
70097,CVE-2026-54920,NONE,NONE,NONE
72353,CVE-2026-55891,NONE,NONE,NONE
79727,CVE-2026-55630,NONE,NONE,NONE
82643,CVE-2026-49292,NONE,NONE,NONE


**Conclusion**

Je garde les 8 failles avec un score à 0. Elles gardent leur gravité NONE.
- C'est une vraie note donnée par un déclarant.
- Elles sont très peu nombreuses (8 sur 84 602 notes v3.1).

## 3. Décisions de nettoyage pour le script final

| # | Sujet | Règle retenue |
|---|---|---|
| 1 | Version de la note | v3.1 pour tout le dashboard. Le CSV garde aussi la v4.0. |
| 2 | Failles sans note v3.1 | Gardées pour les comptages. Exclues des graphiques de gravité et de critères d'attaque. Une carte donne leur part (10,3 % hors rejetées). |
| 3 | Failles rejetées | Exclues de toutes les analyses (2 458 failles). Elles n'ont ni note ni produit ni type de défaut. |
| 4 | Notes Primary ou Secondary | Primary si elle existe. Sinon la première Secondary. |
| 5 | Éditeurs et produits | Vraie table avec une ligne par produit. Code HTML décodé. Espaces retirés. Éditeur vide ou n/a ou N/A ou Unknown devient « non renseigné ». Éditeurs regroupés en minuscules. Doublons supprimés. Failles distinctes comptées. |
| 6 | Types de défaut | Vraie table avec une ligne par type de défaut. Doublons supprimés. Types vagues et failles sans type deviennent « non précisé ». Table `cwe_noms` depuis le catalogue MITRE. |
| 7 | Scores à 0 | Gardés avec la gravité NONE (8 failles). |

À prévoir pour le script final.
- Choisir où exclure les failles rejetées (dans le script ou dans les vues SQL).
- Mettre à jour les failles déjà stockées quand leur statut ou leur note change.
- Traduire en français les colonnes.